# VIB (soft bottleneck) probe

"Starve delta" via a Variational Information Bottleneck: stochastic delta + KL penalty,
keeping the original token-level architecture. Success = the **specialization gap**
(novel vs shared full-effect; baseline = -0.14) flips positive.

Baseline `wiki_model.pt` is untouched; this saves to `wiki_model_vib.pt`.

`beta` is the key knob. Watch `kl=` in Cell 2: it should be non-trivial but not explode
ppl. Requires: GPU ON, Internet ON.

In [ ]:
# Cell 1 - install + clone/pull repo
!pip install transformers scikit-learn scipy datasets -q

import os, sys
from pathlib import Path

REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')

sys.path.insert(0, str(REPO / 'delta_system'))
print('Files:', sorted([f.name for f in (REPO/'delta_system').glob('*.py')]))

In [ ]:
# Cell 2 - train G with VIB (saves to wiki_model_vib.pt). ~30-60 min on T4.
# Watch 'kl=' : non-trivial = bottleneck biting; if ppl explodes, beta is too high.
import subprocess, sys

cmd = [
    sys.executable,
    '/kaggle/working/multihop-retrieval/delta_system/train_vib.py',
    '--steps', '2000',
    '--beta',  '0.1',
]
r = subprocess.run(cmd)
print('exit code', r.returncode)

In [ ]:
# Cell 3 - eval the VIB checkpoint (NOTE the --vib flag). Look at:
#   [1] novel vs shared full-effect -> specialization gap
#   [4] A-dependence fraction (bonus)
import subprocess, sys

cmd = [
    sys.executable,
    '/kaggle/working/multihop-retrieval/delta_system/insertion_cloze_eval.py',
    '--ckpt', '/kaggle/working/checkpoints/wiki_model_vib.pt',
    '--vib',
    '--n',    '500',
]
r = subprocess.run(cmd)
print('exit code', r.returncode)